# 03 — Conversational Evaluation

Multi-turn chat evaluation with `ConversationalTestCase`.

We cover:

* Building a transcript from `Turn` objects.
* Role-adherence and conversation-completeness heuristics.
* Per-turn judged scoring with an offline judge.

## 1. Build a multi-turn test case

In [ ]:
from checkllm.conversation import ConversationalTestCase, Turn

tc = ConversationalTestCase(
    turns=[
        Turn(role="system", content="You are a helpful support agent."),
        Turn(role="user", content="My package hasn't arrived."),
        Turn(role="assistant", content="I'm sorry about that. Could you share your order number?"),
        Turn(role="user", content="Sure, it is #A-123."),
        Turn(
            role="assistant",
            content="Thanks! Your order shipped yesterday and should arrive tomorrow.",
        ),
    ]
)

print("# turns      :", tc.turn_count)
print("first user  :", tc.first_user_message)
print("last reply  :", tc.last_response)
print()
print(tc.format_transcript())

## 2. Deterministic turn-level checks

In [ ]:
assistant_turns = tc.assistant_turns

# Every assistant turn must acknowledge the user; simple containment proxy.
has_apology = any("sorry" in t.content.lower() for t in assistant_turns)
asks_order_num = any("order number" in t.content.lower() for t in assistant_turns)

print("expressed empathy:", has_apology)
print("collected info  :", asks_order_num)

## 3. Per-turn judged scoring (offline)

In [ ]:
from checkllm.judge import JudgeResponse


class FakeJudge:
    """Deterministic in-process judge used to keep this notebook offline.

    Real runs should swap this for ``OpenAIJudge`` / ``AnthropicJudge`` etc.
    """

    def __init__(self, score: float = 0.9, reasoning: str = "looks good") -> None:
        self._score = score
        self._reasoning = reasoning

    async def evaluate(self, prompt: str, system_prompt: str | None = None) -> JudgeResponse:
        return JudgeResponse(
            score=self._score,
            reasoning=self._reasoning,
            raw_output=str(self._score),
            cost=0.0,
        )


judge = FakeJudge(score=0.9, reasoning="offline stub")

In [ ]:
per_turn_scores = []
for turn in tc.assistant_turns:
    prompt = "Assistant reply: " + turn.content + "\n" "Rate role adherence on 0-1 (support agent)."
    resp = await judge.evaluate(prompt=prompt)
    per_turn_scores.append(resp.score)

print("per-turn     :", per_turn_scores)
print("conv. average:", sum(per_turn_scores) / len(per_turn_scores))

## 4. Switch to a real provider

In [ ]:
# -------------------------------------------------------------------
# SWITCH TO A REAL PROVIDER
# -------------------------------------------------------------------
# Uncomment and replace the FakeJudge above once you have API keys:
#
# from checkllm.judge import OpenAIJudge
# judge = OpenAIJudge(api_key="sk-...", model="gpt-4o-mini")
#
# from checkllm.judge import AnthropicJudge
# judge = AnthropicJudge(api_key="...", model="claude-3-5-sonnet-20241022")
